<a href="https://colab.research.google.com/github/Arthurzhl/NeRF/blob/main/colab/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
    <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://docs.nerf.studio/_images/logo-dark.png">
    <source media="(prefers-color-scheme: light)" srcset="https://docs.nerf.studio/_images/logo.png">
    <img alt="nerfstudio" src="https://docs.nerf.studio/_images/logo.png" width="400">
    </picture>
</p>


# Nerfstudio: A collaboration friendly studio for NeRFs


![GitHub stars](https://img.shields.io/github/stars/nerfstudio-project/nerfstudio?color=gold&style=social)

This colab shows how to train and view NeRFs from Nerfstudio both on pre-made datasets or from your own videos/images.

\\

Credit to [NeX](https://nex-mpi.github.io/) for Google Colab format.

## Frequently Asked Questions

*  **Downloading custom data is stalling (no output):**
    * This is a bug in Colab. The data is processing, but may take a while to complete. You will know processing completed if `data/nerfstudio/custom_data/transforms.json` exists. Terminating the cell early will result in not being able to train.
*  **Processing custom data is taking a long time:**
    * The time it takes to process data depends on the number of images and its resolution. If processing is taking too long, try lowering the resolution of your custom data.
*  **Error: Data processing did not complete:**
    * This means that the data processing script did not fully complete. This could be because there were not enough images, or that the images were of low quality. We recommend images with little to no motion blur and lots of visual overlap of the scene to increase the chances of successful processing.
*   **Training is not showing progress**:
    * The lack of output is a bug in Colab. You can see the training progress from the viewer.
* **Viewer Quality is bad / Low resolution**:
    * This may be because more GPU is being used on training that rendering the viewer. Try pausing training or decreasing training utilization.
* **WARNING: Running pip as the 'root' user...:**:
    * This and other pip warnings or errors can be safely ignored.
* **Other problems?**
    * Feel free to create an issue on our [GitHub repo](https://github.com/nerfstudio-project/nerfstudio).


In [4]:
# 1. Install Ninja (required for fast C++ builds)
!pip install ninja

# 2. Set environment variables to bypass common build errors
import os
# '75' targets the T4 GPU; '80' or '86' for A100/L4 if you have Colab Pro
os.environ["TCNN_CUDA_ARCHITECTURES"] = "75"
os.environ["CC"] = "gcc"
os.environ["CXX"] = "g++"

# 3. Install Tiny-CUDA-NN directly from source
# The '--no-build-isolation' flag is the secret to fixing the 'subprocess' error
!pip install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch --no-build-isolation

# 4. Install Nerfstudio
!pip install git+https://github.com/nerfstudio-project/nerfstudio.git

# 5. Final verification
import tinycudann as tcnn
import nerfstudio
print("\n ✅ SUCCESS: The environment is ready for your demo!")

  Cloning https://github.com/NVlabs/tiny-cuda-nn/ to /tmp/pip-req-build-fdypxsjs
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/tiny-cuda-nn/ /tmp/pip-req-build-fdypxsjs
  Resolved https://github.com/NVlabs/tiny-cuda-nn/ to commit 2e757bbe781db59c4980d389d7dccbf5edc09669
  Running command git submodule update --init --recursive -q
  Preparing metadata (pyproject.toml) ... done
  Created wheel for tinycudann: filename=tinycudann-2.0-cp312-cp312-linux_x86_64.whl size=16649024 sha256=2be4ecd887b8055b6f08db38b14e8974cd32c6556f34274afb13b3065b8a48db
  Stored in directory: /tmp/pip-ephem-wheel-cache-daerq2b3/wheels/2f/e8/5a/6f5fba4370cba8f29cff0bb004adb2bfbe0a148555d848019a
Successfully built tinycudann
  Cloning https://github.com/nerfstudio-project/nerfstudio.git to /tmp/pip-req-build-ahou93wv
  Running command git clone --filter=blob:none --quiet https://github.com/nerfstudio-project/nerfstudio.git /tmp/pip-req-build-ahou93wv
  Resolved https://github.com

In [ ]:
# @markdown <h1> Downloading and Processing Data</h1>
# @markdown <h3>Pick the preset scene or upload your own images/video</h3>
import glob
import os

from google.colab import files
from IPython.core.display import HTML, display

scene = "\ud83d\uddbc poster"  # @param ['🖼 poster', '🚜 dozer', '🌄 desolation', '📤 upload your images' , '🎥 upload your own video', '🔺 upload Polycam data', '💽 upload your own Record3D data']
scene = " ".join(scene.split(" ")[1:])

if scene == "upload Polycam data":
    %cd /content/
    !mkdir -p /content/data/nerfstudio/custom_data
    %cd /content/data/nerfstudio/custom_data/
    uploaded = files.upload()
    dir = os.getcwd()
    if len(uploaded.keys()) > 1:
        print("ERROR, upload a single .zip file when processing Polycam data")
    dataset_dir = [os.path.join(dir, f) for f in uploaded.keys()][0]
    !ns-process-data polycam --data $dataset_dir --output-dir /content/data/nerfstudio/custom_data/
    scene = "custom_data"
elif scene == "upload your own Record3D data":
    display(HTML("<h3>Zip your Record3D folder, and upload.</h3>"))
    display(
        HTML(
            '<h3>More information on Record3D can be found <a href="https://docs.nerf.studio/en/latest/quickstart/custom_dataset.html#record3d-capture" target="_blank">here</a>.</h3>'
        )
    )
    %cd /content/
    !mkdir -p /content/data/nerfstudio/custom_data
    %cd /content/data/nerfstudio/custom_data/
    uploaded = files.upload()
    dir = os.getcwd()
    preupload_datasets = [os.path.join(dir, f) for f in uploaded.keys()]
    record_3d_zipfile = preupload_datasets[0]
    !unzip $record_3d_zipfile -d /content/data/nerfstudio/custom_data
    custom_data_directory = glob.glob("/content/data/nerfstudio/custom_data/*")[0]
    !ns-process-data record3d --data $custom_data_directory --output-dir /content/data/nerfstudio/custom_data/
    scene = "custom_data"
elif scene in ["upload your images", "upload your own video"]:
    display(HTML("<h3>Select your custom data</h3>"))
    display(HTML("<p/>You can select multiple images by pressing ctrl, cmd or shift and click.<p>"))
    display(
        HTML(
            "<p/>Note: This may take time, especially on higher resolution inputs, so we recommend to download dataset after creation.<p>"
        )
    )
    !mkdir -p /content/data/nerfstudio/custom_data
    if scene == "upload your images":
        !mkdir -p /content/data/nerfstudio/custom_data/raw_images
        %cd /content/data/nerfstudio/custom_data/raw_images
        uploaded = files.upload()
        dir = os.getcwd()
    else:
        %cd /content/data/nerfstudio/custom_data/
        uploaded = files.upload()
        dir = os.getcwd()
    preupload_datasets = [os.path.join(dir, f) for f in uploaded.keys()]
    del uploaded
    %cd /content/

    if scene == "upload your images":
        !ns-process-data images --data /content/data/nerfstudio/custom_data/raw_images --output-dir /content/data/nerfstudio/custom_data/
    else:
        video_path = preupload_datasets[0]
        !ns-process-data video --data $video_path --output-dir /content/data/nerfstudio/custom_data/

    scene = "custom_data"
else:
    %cd /content/
    !ns-download-data nerfstudio --capture-name=$scene

print("Data Processing Succeeded!")

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# 1. Create a local folder for fast access
!mkdir -p /content/local_images

# 2. Copy the zip from your Google Drive to local memory
!cp /content/drive/MyDrive/toy_project/my_image.zip /content/local_images/

# 3. Unzip the file locally
!unzip -q /content/local_images/my_image.zip -d /content/local_images/

# 4. Cleanup: Remove the zip from local memory to save space (it's still safe on your Drive)
!rm /content/local_images/my_image.zip

print("✅ Extraction complete! Your images are now in /content/local_images/")

✅ Extraction complete! Your images are now in /content/local_images/


In [10]:
# @markdown <h1>Start Training</h1>

%cd /content
!pip install colab-xterm
%load_ext colabxterm
%env TERM=xterm
import os  # Added to ensure os is imported
from IPython.display import clear_output

# --- ADD THIS LINE ---
scene = "custom_data"
# ---------------------

clear_output(wait=True)
if os.path.exists(f"data/nerfstudio/{scene}/transforms.json"):
    print(
        "\033[1m"
        + "Copy and paste the following command into the terminal window that pops up under this cell."
        + "\033[0m"
    )
    print(
        f"ns-train nerfacto --viewer.websocket-port 7007 --viewer.make-share-url True nerfstudio-data --data data/nerfstudio/{scene} --downscale-factor 4"
    )
    print()
    %xterm
else:
    from IPython.core.display import HTML, display
    display(HTML('<h3 style="color:red">Error: Data processing did not complete</h3>'))
    display(HTML(f"<h3>Checked path: data/nerfstudio/{scene}/transforms.json</h3>"))
    display(HTML("<h3>Please run the 'Data Processing' cell first.</h3>"))

In [13]:
# This step takes 5-10 minutes depending on the number of images.
!ns-process-data images --data /content/local_images --output-dir /content/data/nerfstudio/custom_data


[15:49:03] 🎉 Done copying images with prefix 'frame_'.                                        ]8;id=649041;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=511656;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/process_data/process_data_utils.py#348\348]8;;\
(    ● ) Copying images...
──────────────────────────────────────────────  💀 💀 💀 ERROR 💀 💀 💀  ───────────────────────────────────────────────
Error running command: colmap feature_extractor --database_path /content/data/nerfstudio/custom_data/colmap/database.db 
--image_path /content/data/nerfstudio/custom_data/images --ImageReader.single_camera 1 --ImageReader.camera_model OPENCV
--SiftExtraction.use_gpu 1
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
qt.qpa.xcb: could not connect to display 
qt.qpa.plugin: Could not load the Qt platform plugin "xcb" in "" even th

In [15]:
!ns-train vanilla-nerf --data /content/data/nerfstudio/custom_data --viewer.make-share-url True

2026-03-05 15:50:52.082593: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772725852.269465   33534 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772725852.321469   33534 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772725852.721979   33534 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772725852.722021   33534 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772725852.722025   33534 computation_placer.cc:177] computation placer alr

In [16]:
# @title # Render Video { vertical-output: true }
# @markdown <h3>Export the camera path from within the viewer, then run this cell.</h3>
# @markdown <h5>The rendered video should be at renders/output.mp4!</h5>


base_dir = "/content/outputs/unnamed/nerfacto/"
training_run_dir = base_dir + os.listdir(base_dir)[0]

from IPython.core.display import HTML, display

display(HTML("<h3>Upload the camera path JSON.</h3>"))
%cd $training_run_dir
uploaded = files.upload()
uploaded_camera_path_filename = list(uploaded.keys())[0]

config_filename = training_run_dir + "/config.yml"
camera_path_filename = training_run_dir + "/" + uploaded_camera_path_filename
camera_path_filename = camera_path_filename.replace(" ", "\\ ").replace("(", "\\(").replace(")", "\\)")

%cd /content/
!ns-render camera-path --load-config $config_filename --camera-path-filename $camera_path_filename --output-path renders/output.mp4

FileNotFoundError: [Errno 2] No such file or directory: '/content/outputs/unnamed/nerfacto/'